# SignBridge — Notebook 2: YOLO Person, Hand & Face Detection

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/joshiaradhya/SignBridge/blob/main/notebooks/02_yolo_detection.ipynb)

This notebook:
1. Samples representative video frames from sign language datasets.
2. Formats dataset annotations in YOLO format (`person`, `hand`, `face`).
3. Fine-tunes a lightweight **YOLOv8n** model on GPU.
4. Evaluates mAP metrics and exports an optimized ONNX model.

In [ ]:
# Step 1: Environment Setup
!git clone https://github.com/joshiaradhya/SignBridge.git /content/SignBridge || (cd /content/SignBridge && git pull)
%cd /content/SignBridge
!pip install -q -r ml/colab-requirements.txt
!pip install -q ultralytics

In [ ]:
# Step 2: Sample video frames for YOLO dataset
from pathlib import Path
from ml.signbridge.datasets import sample_yolo_frames, export_yolo_yaml

video_paths = list(Path('/content/drive/MyDrive/SignBridge_Data').rglob('*.mp4'))[:100]
yolo_root = Path('data/yolo_dataset')

if video_paths:
    stats = sample_yolo_frames(video_paths, yolo_root, frames_per_video=3)
    export_yolo_yaml(yolo_root / 'data.yaml', yolo_root)
    print(f'Sampled frames: {stats}')
else:
    print('No videos found in Drive data path. Provide video files to sample frames.')

In [ ]:
# Step 3: Train YOLOv8n detector on GPU
from ultralytics import YOLO

# Initialize lightweight nano model
model = YOLO('yolov8n.pt')

# Train on sampled frames
results = model.train(
    data='data/yolo_dataset/data.yaml',
    epochs=30,
    imgsz=640,
    batch=16,
    device=0,
    project='runs/yolo_sign',
    name='person_hand_face'
)

In [ ]:
# Step 4: Validate and Export to ONNX
metrics = model.val()
print('mAP50-95:', metrics.box.map)

# Export to ONNX
onnx_file = model.export(format='onnx', dynamic=True, opset=17)
print('Exported ONNX detector to:', onnx_file)
!cp {onnx_file} /content/drive/MyDrive/SignBridge_Data/sign_yolo_v1.onnx 2>/dev/null || true